# Train YOLOv8 — Transformer Parts (transformer, wire)
Runtime → Change runtime type → **GPU** before running.

This trains from a **labelImg** dataset (YOLO `.txt` labels).

In [ ]:
!pip install -q ultralytics

Upload your `dataset.zip`. It should contain `train/` and `valid/` at its root, each with `images/` and `labels/` subfolders (see `docs/annotation-guide.md`).

In [ ]:
import zipfile, os
from google.colab import files
uploaded = files.upload()  # pick your dataset.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content/dataset")
print("extracted:", os.listdir("/content/dataset"))

In [ ]:
# Write the dataset config. names order MUST match labelImg classes.txt.
data_yaml = "/content/dataset/data.yaml"
with open(data_yaml, "w") as f:
    f.write(
        "path: /content/dataset\n"
        "train: train/images\n"
        "val: valid/images\n"
        "nc: 2\n"
        "names: ['transformer', 'wire']\n"
    )
print(open(data_yaml).read())

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")  # nano: small, fast, enough for 2 classes
model.train(
    data=data_yaml,
    # epochs/imgsz: standard starting point; batch=16 fits a Colab T4's VRAM.
    epochs=100, imgsz=640, patience=20, batch=16,
)

In [ ]:
metrics = model.val()
print("mAP50-95:", metrics.box.map)
print("mAP50:   ", metrics.box.map50)

In [ ]:
# Optional: eyeball predictions on the validation images (saved under runs/detect/predict).
model.predict("/content/dataset/valid/images", conf=0.25, save=True)

In [ ]:
# Download the trained weights to your machine.
from google.colab import files
files.download("runs/detect/train/weights/best.pt")

Put the downloaded `best.pt` into your project's `models/` folder, then run the API: `THERMAL_WEIGHTS=models/best.pt uvicorn api:app --reload`.